# WV County Overdose Deaths: Step 1 (Prepare the data)
Source: 2025 County Health Rankings West Virginia data (v3), CHR&R.

In [2]:
import pandas as pd, numpy as np
np.random.seed(42)

PATH = "data/2025_County_Health_Rankings_West_Virginia_Data_-_v3.xlsx"
sel = pd.read_excel(PATH, sheet_name="Select Measure Data", header=1)
add = pd.read_excel(PATH, sheet_name="Additional Measure Data", header=1)
print(sel.shape, add.shape)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Merge the two sheets by county FIPS code
a = sel[["FIPS", "County", "% Unemployed", "Mental Health Provider Rate"]]
b = add[["FIPS", "Drug Overdose Mortality Rate", "Median Household Income", "% Frequent Mental Distress"]]
df = a.merge(b, on="FIPS", how="inner")

# Remove the statewide row (it has no county name)
df = df[df["County"].notna()].copy()
print(df.shape)  # 55 counties

(55, 7)


In [ ]:
# Rename columns and keep six: county + five measures
df = df.rename(columns={
    "County": "county",
    "Drug Overdose Mortality Rate": "overdose_rate",
    "% Unemployed": "unemployment",
    "Median Household Income": "median_income",
    "% Frequent Mental Distress": "mental_distress",
    "Mental Health Provider Rate": "mh_providers_rate",
})[["county", "overdose_rate", "unemployment", "median_income", "mental_distress", "mh_providers_rate"]]

# Counties with no overdose value (fewer than 10 deaths in 3 years)
removed = df.loc[df["overdose_rate"].isna(), "county"].tolist()
print(len(removed), "counties removed:", removed)

df = df.dropna(subset=["overdose_rate"]).reset_index(drop=True)
print(df.shape)  # expect (45, 6)
assert df.shape == (45, 6)
assert df.isna().sum().sum() == 0

10 counties removed: ['Calhoun', 'Doddridge', 'Gilmer', 'Pendleton', 'Pleasants', 'Ritchie', 'Roane', 'Tucker', 'Webster', 'Wirt']
(45, 6)


In [ ]:
df.to_csv("data/clean_wv.csv", index=False)
df.head()

,county,overdose_rate,unemployment,median_income,mental_distress,mh_providers_rate
0,Barbour,38.034062,4.844339,51353,23.7,123.55313
1,Berkeley,77.313310,3.403996,78995,19.5,207.64120
2,Boone,108.944501,4.205156,48034,23.9,43.74028
3,Braxton,68.180626,5.063291,50038,24.0,98.66798
4,Brooke,68.653028,5.018107,49187,21.7,46.78800


## Data dictionary
| Column | Meaning | Unit | Period |
| --- | --- | --- | --- |
| county | County name | text | n/a |
| overdose_rate | Drug poisoning deaths | per 100,000 people | 2020-2022 |
| unemployment | Population 16+ unemployed but seeking work | % | 2023 |
| median_income | Income at which half of households earn more and half earn less | US dollars | 2023 and 2019-2023 |
| mental_distress | Adults reporting 14+ poor mental health days per month (age-adjusted) | % | 2022 |
| mh_providers_rate | Mental health providers | per 100,000 people | 2024 |